# 02c — Unsupervised Learning

No labels. No "right answer" to learn from.  
Instead: discover **structure, patterns, and groupings** in the data.

```
Supervised:    X, y ──► Model ──► Predict y
Unsupervised:  X    ──► Model ──► Discover structure
```

Two main tasks:
- **Clustering**: group similar data points together
- **Dimensionality Reduction**: compress features while preserving information

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_blobs, make_moons, load_digits
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

---
## 1 — K-Means Clustering

The most popular clustering algorithm. Dead simple:

1. **Pick k random centroids**
2. **Assign** each point to its nearest centroid
3. **Update** centroids to the mean of assigned points
4. **Repeat** until centroids stop moving

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for step, ax in enumerate(axes.ravel()):
    if step == 0:
        ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c='gray', alpha=0.5, s=20)
        np.random.seed(0)
        init_centers = X_blobs[np.random.choice(len(X_blobs), 4, replace=False)]
        ax.scatter(init_centers[:, 0], init_centers[:, 1], c=colors, s=200, marker='X', edgecolors='k', linewidths=2, zorder=5)
        ax.set_title('Step 0: Random centroids')
    else:
        km = KMeans(n_clusters=4, init=init_centers, n_init=1, max_iter=step, random_state=42)
        km.fit(X_blobs)
        labels = km.labels_
        for i in range(4):
            mask = labels == i
            ax.scatter(X_blobs[mask, 0], X_blobs[mask, 1], c=colors[i], alpha=0.5, s=20)
        ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c=colors, s=200, marker='X', edgecolors='k', linewidths=2, zorder=5)
        ax.set_title(f'Step {step}: Assign & update')

plt.suptitle('K-Means — Centroids converge to cluster centers', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
km_final.fit(X_blobs)

plt.figure(figsize=(8, 6))
for i in range(4):
    mask = km_final.labels_ == i
    plt.scatter(X_blobs[mask, 0], X_blobs[mask, 1], c=colors[i], alpha=0.6, s=30, label=f'Cluster {i}')
plt.scatter(km_final.cluster_centers_[:, 0], km_final.cluster_centers_[:, 1],
            c='black', s=200, marker='X', zorder=5, label='Centroids')
plt.title('K-Means Final Result')
plt.legend()
plt.show()

### The Elbow Method — Choosing k

Plot **inertia** (sum of squared distances to nearest centroid) vs k.  
Look for the "elbow" where adding more clusters stops helping much.

In [ ]:
inertias = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=4, color='r', linestyle='--', alpha=0.5, label='Elbow at k=4')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method — Find the sweet spot')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 2 — Hierarchical Clustering

Build a tree of clusters (dendrogram).  
**Agglomerative** (bottom-up): start with each point as its own cluster, merge the closest pairs.

Advantage over K-Means: don't need to specify k upfront — just cut the dendrogram.

In [ ]:
X_small, _ = make_blobs(n_samples=50, centers=3, cluster_std=1.0, random_state=42)

Z = linkage(X_small, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dendrogram(Z, ax=axes[0], truncate_mode='lastp', p=20, leaf_font_size=8)
axes[0].set_title('Dendrogram — Cut at desired height to get clusters')
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Distance')
axes[0].axhline(y=8, color='r', linestyle='--', label='Cut here → 3 clusters')
axes[0].legend()

hc = AgglomerativeClustering(n_clusters=3)
hc_labels = hc.fit_predict(X_small)
for i in range(3):
    mask = hc_labels == i
    axes[1].scatter(X_small[mask, 0], X_small[mask, 1], s=50, edgecolors='k', linewidths=0.5, label=f'Cluster {i}')
axes[1].set_title('Hierarchical Clustering Result')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3 — DBSCAN

**Density-Based** clustering. Finds clusters of arbitrary shape.  
Two parameters:
- **eps**: radius of neighborhood
- **min_samples**: minimum points to form a dense region

Points in sparse regions are labeled as **noise** (outliers).

**When K-Means fails, DBSCAN shines:**

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km_moons.fit_predict(X_moons)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_labels, cmap='RdBu', edgecolors='k', linewidths=0.5)
axes[0].set_title('K-Means FAILS on non-spherical clusters')

db = DBSCAN(eps=0.2, min_samples=5)
db_labels = db.fit_predict(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db_labels, cmap='RdBu', edgecolors='k', linewidths=0.5)
axes[1].set_title('DBSCAN handles it perfectly')

plt.suptitle('Non-spherical clusters: K-Means vs DBSCAN', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
n_noise = np.sum(db_labels == -1)
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
print(f'DBSCAN found {n_clusters} clusters and {n_noise} noise points')

---
## 4 — PCA (Principal Component Analysis)

Reduce dimensions while keeping maximum variance.  
Projects data onto the directions of greatest spread.

PCA uses **eigenvalues** from Topic 00! The eigenvectors of the covariance matrix are the principal components.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)
n = 200
X_3d = np.dot(np.random.randn(n, 3), [[2, 0, 0], [0, 1, 0], [0, 0, 0.2]])
rotation = np.array([[0.6, -0.8, 0], [0.8, 0.6, 0], [0, 0, 1]])
X_3d = X_3d @ rotation

pca_3d = PCA(n_components=2)
X_2d = pca_3d.fit_transform(X_3d)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], alpha=0.5, s=10)
ax1.set_title('Original 3D data')

ax2 = fig.add_subplot(122)
ax2.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.5, s=10)
ax2.set_title('Projected to 2D with PCA')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')

plt.tight_layout()
plt.show()

print(f'Explained variance ratio: {pca_3d.explained_variance_ratio_}')
print(f'Total variance kept:      {pca_3d.explained_variance_ratio_.sum():.3f}')

In [ ]:
digits = load_digits()
X_dig, y_dig = digits.data, digits.target

pca_dig = PCA().fit(X_dig)

plt.figure(figsize=(8, 5))
plt.plot(np.cumsum(pca_dig.explained_variance_ratio_), 'b-', linewidth=2)
plt.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.title('PCA — How many components do we need?')
plt.legend()
plt.grid(True, alpha=0.3)

n_95 = np.argmax(np.cumsum(pca_dig.explained_variance_ratio_) >= 0.95) + 1
plt.axvline(x=n_95, color='g', linestyle='--', alpha=0.5, label=f'{n_95} components for 95%')
plt.legend()
plt.show()

print(f'Original dimensions: {X_dig.shape[1]}')
print(f'Components for 95% variance: {n_95}')

In [ ]:
pca_2d = PCA(n_components=2)
X_dig_2d = pca_2d.fit_transform(X_dig)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_dig_2d[:, 0], X_dig_2d[:, 1], c=y_dig, cmap='tab10', s=10, alpha=0.6)
plt.colorbar(scatter, label='Digit')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Digits → 2D with PCA (64 → 2 dimensions)')
plt.show()

---
## 5 — t-SNE

**Non-linear** dimensionality reduction. Better at preserving local structure → tighter clusters in visualization.

t-SNE vs PCA:
- PCA: fast, deterministic, preserves global structure
- t-SNE: slow, stochastic, preserves local structure (better for visualization)

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_dig_tsne = tsne.fit_transform(X_dig)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(X_dig_2d[:, 0], X_dig_2d[:, 1], c=y_dig, cmap='tab10', s=5, alpha=0.6)
axes[0].set_title('PCA — Some overlap between digits')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(X_dig_tsne[:, 0], X_dig_tsne[:, 1], c=y_dig, cmap='tab10', s=5, alpha=0.6)
axes[1].set_title('t-SNE — Much clearer clusters!')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

plt.suptitle('Digits Dataset: PCA vs t-SNE Visualization', fontsize=13)
plt.tight_layout()
plt.show()

---
## Clustering Comparison Summary

| Algorithm | Cluster Shape | Need k? | Handles Noise? | Speed |
|-----------|--------------|---------|----------------|-------|
| K-Means | Spherical | Yes | No | Fast |
| Hierarchical | Any | Cut dendrogram | No | Slow (O(n³)) |
| DBSCAN | Arbitrary | No (eps, min_samples) | Yes | Medium |

## Key Takeaways

| Concept | Remember |
|---------|----------|
| K-Means | Simple, fast, but assumes spherical clusters |
| DBSCAN | Arbitrary shapes, detects outliers, but sensitive to eps |
| PCA | Linear reduction, preserves global structure |
| t-SNE | Non-linear, beautiful visualizations, but slow and non-deterministic |
| Elbow method | Plot inertia vs k, pick the bend |

**Next**: Evaluation & tuning — how to know if your model is actually good →